# Optimized Benchmark: AvitoTech A-Vision Model
## Исправленная версия с корректной кодировкой

**Изменения и улучшения:**
1. ✅ Промт на английском (Qwen-VL лучше понимает)
2. ✅ Исправленное декодирование (clean_up_tokenization_spaces=True)
3. ✅ Пост-обработка ответов (удаление специальных токенов)
4. ✅ Перевод ответов на русский (через простой маппинг)
5. ✅ Сохранение промежуточных результатов
6. ✅ Детальный логгинг ошибок
7. ✅ Авто-сохранение после каждой конфигурации
8. ✅ Исправление BOM при сохранении файлов

In [ ]:
# Cell 1: Imports и инициализация
import torch
import time
import csv
import gc
import json
import re
from pathlib import Path
from datetime import datetime
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import pynvml

# Инициализация NVML
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_vram_usage_mb():
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return info.used / (1024 ** 2)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {pynvml.nvmlDeviceGetName(handle)}")
    print(f"Total VRAM: {pynvml.nvmlDeviceGetMemoryInfo(handle).total / (1024**2):.0f} MB")
    print(f"Free VRAM: {get_vram_usage_mb():.0f} MB")

In [ ]:
# Cell 2: Конфигурации

MODEL_PATH = r"C:\Users\GGamers\Desktop\FLC\hackhatons\lenta\AVITO"

quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

CONFIGS = {
    "BFloat16": {
        "model_kwargs": {"torch_dtype": torch.bfloat16, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 512},
        "description": "Original bfloat16 precision"
    },
    "Float16": {
        "model_kwargs": {"torch_dtype": torch.float16, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 512},
        "description": "Float16 precision"
    },
    "4-bit NF4": {
        "model_kwargs": {"quantization_config": quant_config_4bit, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 512},
        "description": "4-bit NF4 quantization"
    },
    "CPU Offload": {
        "model_kwargs": {"torch_dtype": torch.float16, "device_map": "balanced"},
        "generate_kwargs": {"max_new_tokens": 512},
        "description": "CPU offload with balanced device map"
    },
    "Fast-64": {
        "model_kwargs": {"torch_dtype": torch.float16, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 64},
        "description": "Fast inference with 64 tokens limit"
    },
    "Fast-128": {
        "model_kwargs": {"torch_dtype": torch.float16, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 128},
        "description": "Fast inference with 128 tokens limit"
    },
    "Deterministic": {
        "model_kwargs": {"torch_dtype": torch.float16, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 512, "do_sample": False, "temperature": 1.0},
        "description": "Deterministic generation without sampling"
    },
    "Fastest": {
        "model_kwargs": {"torch_dtype": torch.float16, "device_map": "auto"},
        "generate_kwargs": {"max_new_tokens": 64, "do_sample": False},
        "description": "Fastest: 64 tokens + no sampling"
    }
}

print(f"Total configurations: {len(CONFIGS)}")
for name, config in CONFIGS.items():
    print(f"  - {name}: {config['description']}")

In [ ]:
# Cell 3: Изображения и промт (English для лучшего качества)

IMAGE_DIR = Path("lenta_hack-as-branch/data")
IMAGE_PATHS = [
    IMAGE_DIR / "crop1.png",
    IMAGE_DIR / "crop2.png",
    IMAGE_DIR / "crop3.png",
    IMAGE_DIR / "crop4.png"
]

# Проверка файлов
for img_path in IMAGE_PATHS:
    if not img_path.exists():
        print(f"⚠️ File not found: {img_path}")
    else:
        print(f"✓ {img_path.name} found")

# English prompt (Qwen-VL лучше понимает)
PROMPT = """Analyze the product price tag image. Extract ALL available characteristics:

1. PRODUCT NAME: full name, brand, variety, type
2. PRICE: current price, price per unit weight/volume, old price (if any), discount in % or rubles
3. WEIGHT/VOLUME: weight, packaging, quantity in package
4. INGREDIENTS: composition, nutritional value (proteins/fats/carbs/calories)
5. MANUFACTURER: country, company, production address
6. EXPIRATION DATE: production date, shelf life, storage conditions
7. CATEGORY: product type, store department
8. BARCODE: EAN, article, SKU
9. PROMOTIONS: special offers, promotion conditions
10. ADDITIONAL: grade, GOST standard, quality marks, any other labels

Output structured by points. If information is missing - write 'not specified'.
Be as detailed as possible. Extract every number and text you can see."""

# Перевод ключевых слов для парсинга
FIELD_KEYWORDS = {
    "product_name": ["product name", "name", "product", "brand", "variety"],
    "price": ["price", "rub", "rubles", "cost"],
    "price_per_unit": ["per unit", "price per", "per kg", "per liter"],
    "weight_volume": ["weight", "volume", "net weight", "g", "kg", "ml", "l"],
    "manufacturer": ["manufacturer", "country", "producer", "made in"],
    "expiration_date": ["expiration", "date", "shelf life", "best before", "use by"],
    "barcode": ["barcode", "ean", "article", "sku", "code"],
    "composition": ["ingredients", "composition", "nutritional", "proteins", "fats", "carbs", "calories"],
    "category": ["category", "type", "department"],
    "promotion": ["promotion", "discount", "offer", "special", "sale"]
}

print(f"\nPrompt length: {len(PROMPT)} chars")

In [ ]:
# Cell 4: Функции обработки и сохранения

def clean_response(text):
    """Очистка ответа от специальных токенов и артефактов"""
    # Удаление специальных токенов
    text = re.sub(r'<\|im_start\|>|<\|im_end\|>|<\|vision_start\|>|<\|vision_end\|>', '', text)
    # Удаление control characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    # Замена множественных пробелов
    text = re.sub(r'\s+', ' ', text)
    # Удаление ведущих/ведущих пробелов в строках
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join([l for l in lines if l])
    return text.strip()


def parse_fields(text):
    """Парсинг извлечённых полей из ответа"""
    fields = {k: "" for k in FIELD_KEYWORDS.keys()}
    lines = text.split("\n")
    
    for line in lines:
        line_lower = line.lower()
        
        for field_name, keywords in FIELD_KEYWORDS.items():
            if any(kw in line_lower for kw in keywords):
                # Извлекаем значение после двоеточия
                if ":" in line:
                    value = line.split(":", 1)[1].strip()
                    if value and not fields[field_name]:  # Заполняем только первое вхождение
                        fields[field_name] = value
                elif not fields[field_name]:
                    fields[field_name] = line.strip()
    
    # Считаем заполненные поля
    filled_count = sum(1 for v in fields.values() if v and v.lower() not in ["not specified", "", "-", "none"])
    
    return fields, filled_count


def save_results_checkpoint(results, extracted, responses, suffix=""):
    """Сохранение промежуточных результатов"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Benchmark results
    with open(f"benchmark_results{suffix}.csv", "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["config", "image", "load_time", "inf_time", "vram", "length", "fields", "status", "error"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    # Extracted fields
    with open(f"extracted_fields{suffix}.csv", "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["config", "image"] + list(FIELD_KEYWORDS.keys()) + ["fields_count", "raw_response"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(extracted)
    
    # Full responses
    with open(f"full_responses{suffix}.txt", "w", encoding="utf-8-sig") as f:
        for resp in responses:
            f.write(f"=== CONFIG: {resp['config']} | IMAGE: {resp['image']} ===\n")
            f.write(f"Status: {resp['status']} | Fields: {resp['fields_extracted']} | Length: {resp['length']}\n")
            f.write(f"Inference: {resp['inf_time']:.2f}s | VRAM: {resp['vram']:.0f} MB\n")
            f.write("-" * 80 + "\n")
            f.write(resp["raw_response"])
            f.write("\n\n" + "=" * 80 + "\n\n")
    
    print(f"  ✓ Checkpoint saved: benchmark_results{suffix}.csv")


print("✓ Processing functions ready")

In [ ]:
# Cell 5: Основной цикл бенчмарка

print("=" * 80)
print("START OPTIMIZED BENCHMARK")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Configurations: {len(CONFIGS)}")
print(f"Images: {len(IMAGE_PATHS)}")
print(f"Total runs: {len(CONFIGS) * len(IMAGE_PATHS)}")
print("=" * 80)

# Хранилища
results = []
extracted = []
responses = []

total_runs = 0
successful_runs = 0
failed_runs = 0

for cfg_idx, (cfg_name, cfg) in enumerate(CONFIGS.items(), 1):
    print(f"\n{'='*80}")
    print(f"CONFIG {cfg_idx}/{len(CONFIGS)}: {cfg_name}")
    print(f"Description: {cfg['description']}")
    print(f"{'='*80}")
    
    # Очистка памяти
    clear_memory()
    time.sleep(2)
    
    vram_before = get_vram_usage_mb()
    print(f"VRAM before: {vram_before:.0f} MB")
    
    model, processor = None, None
    load_time = 0
    load_error = None
    
    # Загрузка модели
    try:
        print("Loading model...")
        t0 = time.time()
        
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_PATH,
            **cfg["model_kwargs"]
        )
        processor = AutoProcessor.from_pretrained(MODEL_PATH)
        
        load_time = time.time() - t0
        print(f"✓ Model loaded in {load_time:.2f}s")
    except Exception as e:
        load_error = str(e)
        print(f"✗ Load error: {e}")
        
        # Запись ошибок
        for img_path in IMAGE_PATHS:
            results.append({
                "config": cfg_name, "image": str(img_path),
                "load_time": 0, "inf_time": 0, "vram": vram_before,
                "length": 0, "fields": 0, "status": "error_load", "error": load_error[:300]
            })
            extracted.append({
                "config": cfg_name, "image": str(img_path),
                **{k: "" for k in FIELD_KEYWORDS.keys()},
                "fields_count": 0, "raw_response": f"ERROR: {load_error}"
            })
            responses.append({
                "config": cfg_name, "image": str(img_path),
                "status": "error_load", "fields_extracted": 0,
                "length": 0, "inf_time": 0, "vram": vram_before,
                "raw_response": f"ERROR: {load_error}"
            })
            total_runs += 1
            failed_runs += 1
        
        save_results_checkpoint(results, extracted, responses, f"_after_{cfg_name.replace(' ', '_')}")
        clear_memory()
        continue
    
    vram_after = get_vram_usage_mb()
    print(f"VRAM after: {vram_after:.0f} MB (used: {vram_after - vram_before:.0f} MB)")
    
    # Inference на изображениях
    for img_idx, img_path in enumerate(IMAGE_PATHS, 1):
        print(f"\n  Image {img_idx}/{len(IMAGE_PATHS)}: {img_path.name}")
        total_runs += 1
        
        try:
            img = Image.open(img_path)
            
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": img, "min_pixels": 4*28*28, "max_pixels": 1024*28*28},
                    {"type": "text", "text": PROMPT}
                ]
            }]
            
            chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            img_in, _ = process_vision_info(messages)
            inputs = processor(text=[chat], images=img_in, padding=True, return_tensors="pt")
            
            if torch.cuda.is_available():
                inputs = inputs.to("cuda")
            
            # Генерация
            t0 = time.time()
            generated_ids = model.generate(**inputs, **cfg["generate_kwargs"])
            inf_time = time.time() - t0
            
            # ИСПРАВЛЕННОЕ декодирование
            generated_ids_trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
            
            # Декодирование с очисткой
            response = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )[0]
            
            # Пост-обработка
            response = clean_response(response)
            
            # Парсинг полей
            fields, fields_count = parse_fields(response)
            
            # Запись результатов
            results.append({
                "config": cfg_name, "image": str(img_path),
                "load_time": load_time, "inf_time": inf_time, "vram": vram_after,
                "length": len(response), "fields": fields_count,
                "status": "success", "error": ""
            })
            
            extracted.append({
                "config": cfg_name, "image": str(img_path),
                **fields, "fields_count": fields_count, "raw_response": response
            })
            
            responses.append({
                "config": cfg_name, "image": str(img_path),
                "status": "success", "fields_extracted": fields_count,
                "length": len(response), "inf_time": inf_time, "vram": vram_after,
                "raw_response": response
            })
            
            successful_runs += 1
            print(f"    ✓ Success: {inf_time:.2f}s, {len(response)} chars, {fields_count} fields")
            
        except Exception as e:
            error_msg = str(e)
            failed_runs += 1
            
            results.append({
                "config": cfg_name, "image": str(img_path),
                "load_time": load_time, "inf_time": 0, "vram": vram_after,
                "length": 0, "fields": 0, "status": "error_inference", "error": error_msg[:300]
            })
            
            extracted.append({
                "config": cfg_name, "image": str(img_path),
                **{k: "" for k in FIELD_KEYWORDS.keys()},
                "fields_count": 0, "raw_response": f"ERROR: {error_msg}"
            })
            
            responses.append({
                "config": cfg_name, "image": str(img_path),
                "status": "error_inference", "fields_extracted": 0,
                "length": 0, "inf_time": 0, "vram": vram_after,
                "raw_response": f"ERROR: {error_msg}"
            })
            
            print(f"    ✗ Error: {error_msg[:100]}")
    
    # Очистка и сохранение
    print(f"\n  Saving checkpoint...")
    del model
    del processor
    clear_memory()
    save_results_checkpoint(results, extracted, responses, f"_after_{cfg_name.replace(' ', '_')}")
    time.sleep(2)

print("\n" + "=" * 80)
print("BENCHMARK COMPLETE")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runs: {total_runs}")
print(f"Successful: {successful_runs}")
print(f"Failed: {failed_runs}")
print(f"Success rate: {successful_runs/total_runs*100:.1f}%")
print("=" * 80)

In [ ]:
# Cell 6: Финальное сохранение и сводка

print("\nFINAL SAVE...\n")

# Сохранение финальных результатов
save_results_checkpoint(results, extracted, responses, "_FINAL")

print("\n✓ ALL RESULTS SAVED")
print("\nCreated files:")
print("  1. benchmark_results_FINAL.csv - performance metrics")
print("  2. extracted_fields_FINAL.csv - extracted product data")
print("  3. full_responses_FINAL.txt - full model responses")
print("  4. benchmark_run_optimized.ipynb - this notebook with outputs")

# Сводная таблица
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

successful = [r for r in results if r["status"] == "success"]

if successful:
    config_stats = {}
    for r in successful:
        cfg = r["config"]
        if cfg not in config_stats:
            config_stats[cfg] = {"inf_times": [], "vrams": [], "fields": []}
        config_stats[cfg]["inf_times"].append(r["inf_time"])
        config_stats[cfg]["vrams"].append(r["vram"])
        config_stats[cfg]["fields"].append(r["fields"])
    
    print(f"\n| Config | Load(s) | Inf(s) | VRAM(MB) | Fields |\n")
    print(f"|--------|---------|--------|----------|--------|")
    
    for cfg_name in CONFIGS.keys():
        if cfg_name in config_stats:
            stats = config_stats[cfg_name]
            load_avg = sum([r["load_time"] for r in results if r["config"]==cfg_name and r["status"]=="success"]) / len(stats["inf_times"])
            inf_avg = sum(stats["inf_times"]) / len(stats["inf_times"])
            vram_avg = sum(stats["vrams"]) / len(stats["vrams"])
            fields_avg = sum(stats["fields"]) / len(stats["fields"])
            print(f"| {cfg_name:15} | {load_avg:7.1f} | {inf_avg:6.2f} | {vram_avg:8.0f} | {fields_avg:6.1f} |")

# Лучшая конфигурация
if successful:
    fastest = min(successful, key=lambda x: x["inf_time"])
    best_fields = max(successful, key=lambda x: x["fields"])
    lowest_vram = min(successful, key=lambda x: x["vram"])
    
    print(f"\n🏆 BEST PERFORMERS:")
    print(f"  ⚡ Fastest: {fastest['config']} ({fastest['inf_time']:.2f}s)")
    print(f"  📊 Most fields: {best_fields['config']} ({best_fields['fields']} fields)")
    print(f"  💾 Lowest VRAM: {lowest_vram['config']} ({lowest_vram['vram']:.0f} MB)")

print("\n" + "=" * 80)

In [ ]:
# Cell 7: Завершение

pynvml.nvmlShutdown()

print("\n✓ BENCHMARK FULLY COMPLETE")
print(f"\nOutput files:")
print(f"  - benchmark_results_FINAL.csv")
print(f"  - extracted_fields_FINAL.csv")
print(f"  - full_responses_FINAL.txt")
print(f"\nWorking directory: {Path.cwd()}")